# Pemecah kalimat korpus sejarah

In [ ]:
import re
from collections import Counter
from copy import deepcopy

POLA_LINDUNG = [
    r"\b(?:Ir|Dr|dr|Lt|Drs|Dra|Mr|Prof|Moh|Muh|Hj|H|KH|No|no|Jl|hlm|Kab|Kec|Kep|Let|Kol|Letkol|Brigjen|har)\.",
    r"\b(?:[A-Z]\.[ \t]*){2,}",
    r"\b[A-Z]\.(?=[ \t]+[A-Z][a-z])",
    r"(?<=\d)\.(?=\d)",
    r'(?:^|[;:“"(\n])\s*(?:dan\s+)?\d{1,2}\.(?=\s)',
]


def pecah_kalimat(teks):
    """Kembalikan kalimat serta posisi karakter pada teks blok asli."""
    if not isinstance(teks, str):
        raise TypeError("Teks blok harus berupa string.")
    terlindungi = set()
    for pola in POLA_LINDUNG:
        for cocok in re.finditer(pola, teks):
            terlindungi.update(range(cocok.start(), cocok.end()))
    for cocok in re.finditer(r"(?:https?://|www\.)\S+", teks):
        alamat = cocok.group().rstrip(".,!?;:)]}\"'”’")
        terlindungi.update(range(cocok.start(), cocok.start() + len(alamat)))

    hasil = []

    def simpan(awal, akhir):
        while awal < akhir and teks[awal].isspace():
            awal += 1
        while akhir > awal and teks[akhir - 1].isspace():
            akhir -= 1
        if awal < akhir:
            hasil.append({"teks_asli": teks[awal:akhir],
                          "awal_karakter_blok": awal,
                          "akhir_karakter_blok": akhir})

    awal = i = 0
    while i < len(teks):
        if teks[i] not in ".!?" or i in terlindungi:
            i += 1
            continue
        akhir = i + 1
        while akhir < len(teks) and teks[akhir] in ".!?":
            akhir += 1
        sesudah_tanda = akhir
        while akhir < len(teks) and teks[akhir] in "\"'”’)]}»":
            akhir += 1
        berikutnya = akhir
        while berikutnya < len(teks) and teks[berikutnya].isspace():
            berikutnya += 1
        lanjutan_kutipan = (akhir > sesudah_tanda and berikutnya < len(teks)
                            and teks[berikutnya].islower())
        batas = akhir == len(teks) or teks[akhir].isspace()
        if batas and not lanjutan_kutipan:
            simpan(awal, akhir)
            awal = berikutnya
        i = akhir
    simpan(awal, len(teks))
    return hasil


def buat_data_kalimat(dokumen):
    """Ubah bagian/blok JSON v2.2 menjadi daftar kalimat bersumber."""
    if not isinstance(dokumen, list):
        raise ValueError("Input harus berupa daftar artikel, bukan JSON keluaran.")
    semua = []
    id_blok_dilihat = set()
    for d in dokumen:
        for bagian in d["bagian"]:
            for blok in bagian["blok"]:
                bid = blok["id_blok"]
                if bid in id_blok_dilihat:
                    raise ValueError("ID blok duplikat: " + bid)
                id_blok_dilihat.add(bid)
                if not blok.get("layak_retrieval", False):
                    continue
                for nomor, potongan in enumerate(pecah_kalimat(blok["teks"]), 1):
                    teks = potongan["teks_asli"]
                    semua.append({
                        "id_kalimat": f"{bid}-K{nomor:03d}",
                        "id_dokumen": d["id"], "judul_dokumen": d["judul"],
                        "id_bagian": bagian["id_bagian"],
                        "judul_bagian": bagian["judul"],
                        "id_blok": bid, "jenis_blok": blok["jenis"],
                        **potongan,
                        "teks_cari": re.sub(r"\s+", " ", teks).strip().lower(),
                        "layak_retrieval": blok["layak_retrieval"],
                        "indikasi_ejaan_historis": blok.get(
                            "indikasi_ejaan_historis", False
                        ),
                        "url_sumber": d["url_sumber"],
                    })
    assert len({k["id_kalimat"] for k in semua}) == len(semua)
    return semua


def uji_contoh():
    kasus = [
        ("Dr. Andi bertemu Ir. Budi. Mereka mengumpulkan 4.000.000 rupiah. Selesai.",
         ["Dr. Andi bertemu Ir. Budi.", "Mereka mengumpulkan 4.000.000 rupiah.", "Selesai."]),
        ("K.R.T. Arga datang. A. W. S. Budi juga hadir.",
         ["K.R.T. Arga datang.", "A. W. S. Budi juga hadir."]),
        ("Ia berangkat pukul 10.30. Ia tiba pukul 11.00.",
         ["Ia berangkat pukul 10.30.", "Ia tiba pukul 11.00."]),
        ('Ia berkata, “Kita siap.” Mereka berangkat.',
         ['Ia berkata, “Kita siap.”', 'Mereka berangkat.']),
        ('“Merdeka!” serunya. Mereka bersorak.',
         ['“Merdeka!” serunya.', 'Mereka bersorak.']),
        ("Siapa dia? Ia hadir! Semua menunggu.",
         ["Siapa dia?", "Ia hadir!", "Semua menunggu."]),
        ("Lihat https://contoh.id/arsip. Berikutnya ada lampiran.",
         ["Lihat https://contoh.id/arsip.", "Berikutnya ada lampiran."]),
        ("Nama anggota tanpa tanda titik", ["Nama anggota tanpa tanda titik"]),
        ('Asasnya: “1. Persatuan; 2. Kemanusiaan; dan 3. Keadilan”. Itu usulnya.',
         ['Asasnya: “1. Persatuan; 2. Kemanusiaan; dan 3. Keadilan”.', 'Itu usulnya.']),
        ('Judul no. 14 ada di Kab. Arga. Ia membaca.',
         ['Judul no. 14 ada di Kab. Arga.', 'Ia membaca.']),
        ("   ", []),
    ]
    for teks, harapan in kasus:
        hasil = [k["teks_asli"] for k in pecah_kalimat(teks)]
        assert hasil == harapan, (teks, hasil, harapan)
    return len(kasus)


## 2. Periksa contoh singkatan, angka, kutipan, dan daftar


In [ ]:
jumlah_uji = uji_contoh()
print(f"Lolos {jumlah_uji} contoh uji.")
contoh = "K.R.T. Arga bertemu Dr. Andi. Mereka membawa 4.000.000 rupiah."
for nomor, potongan in enumerate(pecah_kalimat(contoh), 1):
    print(nomor, potongan["teks_asli"])


Lolos 11 contoh uji.
1 K.R.T. Arga bertemu Dr. Andi.
2 Mereka membawa 4.000.000 rupiah.


## 3. Unggah satu file korpus JSON



In [ ]:
import hashlib
import json
from pathlib import Path
from google.colab import files

unggahan = files.upload()
if len(unggahan) != 1:
    raise ValueError("Unggah tepat satu file JSON korpus.")
nama_input, isi_input = next(iter(unggahan.items()))
dokumen = json.loads(isi_input.decode("utf-8-sig"))
if not isinstance(dokumen, list) or not dokumen:
    raise ValueError("Input harus berupa daftar artikel yang tidak kosong.")
for artikel in dokumen:
    if not isinstance(artikel, dict) or "id" not in artikel or "bagian" not in artikel:
        raise ValueError("Struktur input tidak sesuai korpus artikel.")
print("File:", nama_input)
print("Jumlah dokumen:", len(dokumen))


Saving korpus_sejarah_indonesia_aktif_ringkas (1).json to korpus_sejarah_indonesia_aktif_ringkas (1) (1).json
File: korpus_sejarah_indonesia_aktif_ringkas (1) (1).json
Jumlah dokumen: 21


## 4. Proses korpus dan simpan hasil



In [ ]:
import json
import re
from pathlib import Path

kalimat = buat_data_kalimat(dokumen)

blok_sumber = {
    b["id_blok"]: b
    for d in dokumen
    for s in d["bagian"]
    for b in s["blok"]
}

kelompok_blok = {}

for k in kalimat:
    b = blok_sumber[k["id_blok"]]

    awal = k["awal_karakter_blok"]
    akhir = k["akhir_karakter_blok"]

    assert b["teks"][awal:akhir] == k["teks_asli"], (
        f"Posisi karakter tidak cocok: {k['id_kalimat']}"
    )

    kelompok_blok.setdefault(k["id_blok"], []).append(k)

for bid, kumpulan in kelompok_blok.items():
    gabungan = "".join(k["teks_asli"] for k in kumpulan)

    assert (
        re.sub(r"\s", "", gabungan)
        == re.sub(r"\s", "", blok_sumber[bid]["teks"])
    ), f"Ada teks yang hilang atau berubah: {bid}"

blok_aktif = {
    bid
    for bid, b in blok_sumber.items()
    if b.get("layak_retrieval", False) and b["teks"].strip()
}

assert set(kelompok_blok) == blok_aktif, (
    "Ada blok aktif yang belum diproses."
)

hasil = {
    "metadata": {
        "versi_format_kalimat": "1.0",
        "file_sumber": nama_input,
        "jumlah_dokumen": len(dokumen),
        "jumlah_blok_diproses": len(kelompok_blok),
        "jumlah_kalimat": len(kalimat),
        "dokumen_sumber": [
            {
                "id": d["id"],
                "judul": d["judul"],
            }
            for d in dokumen
        ],
    },
    "kalimat": kalimat,
}

nama_output = "korpus_kalimat.json"

Path(nama_output).write_text(
    json.dumps(hasil, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Jumlah kalimat:", len(kalimat))
print("Jumlah blok:", len(kelompok_blok))
print("Tersimpan:", "korpus_kalimat.json")

Jumlah kalimat: 3140
Jumlah blok: 1133
Tersimpan: korpus_kalimat.json


## 5. Melihat beberapa hasil


In [ ]:
for k in kalimat[:5]:
    print(k["id_kalimat"])
    print(k["teks_asli"])
    print()

contoh_d14 = [
    k for k in kalimat
    if k["id_blok"] == "D14-B0007"
]

print(
    "Blok D14-B0007 menghasilkan",
    len(contoh_d14),
    "potongan."
)

for k in contoh_d14:
    print(k["id_kalimat"], "|", k["teks_asli"])

D01-B0001-K001
Kekaisaran Jepang menduduki Hindia Belanda (sekarang Indonesia) selama Perang Dunia II, mulai Maret 1942 hingga setelah berakhirnya perang pada September 1945.

D01-B0002-K001
Pada bulan Mei 1940, Jerman menduduki Belanda, dan status darurat perang diberlakukan di Hindia Belanda.

D01-B0002-K002
Menyusul kegagalan perundingan antara pihak berwenang Belanda dan Jepang, aset-aset Jepang di wilayah tersebut dibekukan.

D01-B0002-K003
Belanda menyatakan perang terhadap Jepang setelah serangan terhadap Pearl Harbor pada tanggal 7 Desember 1941.

D01-B0002-K004
Invasi Jepang ke Hindia Belanda dimulai pada tanggal 10 Januari 1942, dan Angkatan Darat Kekaisaran Jepang berhasil menguasai seluruh wilayah koloni tersebut dalam waktu kurang dari tiga bulan.

Blok D14-B0007 menghasilkan 10 potongan.
D14-B0007-K001 | Bandung sengaja dibakar oleh TRI dan rakyat setempat dengan maksud agar Sekutu tidak dapat menggunakan Bandung sebagai markas strategis militer.
D14-B0007-K002 | Di mana-

## 6. Unduh JSON


In [ ]:
files.download(nama_output)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>